# Supplementary Figure S3 — PR-AUC vs order: robustness check (Top-50)

Identical to `R2_order_effects/R2_A_fig3.ipynb` but with `N_TOP = 50`.
Shows that the order-effect conclusions are stable across the top-50
n-plets rather than just the single best.

**Inputs**: `results/R1_B_nplet_eval_MA.csv`, `results/R1_B_nplet_eval_DBS.csv`

In [ ]:
# Re-use exactly the same code as R2_A_fig3, with N_TOP changed.
# Run R2_A_fig3 and change the parameter at the top, OR run this notebook
# which sets N_TOP=50 before importing the shared logic.

N_TOP = 50  # ← robustness check

from pathlib import Path
import os

def ensure_project_root(target_name: str = "high-order-anesthesia") -> Path:
    cwd = Path.cwd().resolve()
    if cwd.name == target_name:
        return cwd
    for parent in cwd.parents:
        if parent.name == target_name:
            os.chdir(parent)
            return parent
    raise RuntimeError(f"Could not find '{target_name}' in path.")

ROOT = ensure_project_root()
print(f"Now in: {ROOT.name}")

In [ ]:
import logging
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from ast import literal_eval
from thoi.measures.gaussian_copula import nplets_measures
from src.hoi_anesthesia.io import load_covariance_dict

results_path = "results"
data_path    = "data"
device       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
FONTSIZE     = 12
LWIDTH       = 2
MSIZE        = 7
HOI_MEASURE  = "O"

plt.rcParams.update({"font.size": FONTSIZE})
sns.set_theme(style="white")
all_covs = load_covariance_dict(f"{data_path}/covariance_matrices_gc.h5")

In [ ]:
def to_nplet_tuple(x):
    from ast import literal_eval
    if isinstance(x, (list, tuple, np.ndarray)):
        return tuple(int(i) for i in x)
    if isinstance(x, str):
        return tuple(int(i) for i in literal_eval(x))
    raise TypeError(f"Unexpected type: {type(x)}")

def build_X_for_dataset(dataset, all_covs):
    if dataset == "MA":
        cons = {"MA": ["MA_awake"]}
        nonr = {"MA": ["deep_propofol", "ketamine", "moderate_propofol", "ts_selv2", "ts_selv4"]}
    else:
        cons = {"DBS": ["DBS_awake", "ts_on_5V"]}
        nonr = {"DBS": ["ts_off", "ts_on_3V_control", "ts_on_5V_control"]}
    rows_c = [cov for ds, sts in cons.items() for st in sts for cov in all_covs[ds][st]]
    rows_n = [cov for ds, sts in nonr.items() for st in sts for cov in all_covs[ds][st]]
    return torch.tensor(np.array(rows_c + rows_n)), len(rows_c), len(rows_n)

In [ ]:
nplet_eval = {}
for dataset in ["MA", "DBS"]:
    df = pd.read_csv(f"{results_path}/R1_B_nplet_eval_{dataset}.csv")
    df = df.query("measure in ['O', 'FC_mean_z']")
    df["nplet"] = df["optimal_nplet"].apply(to_nplet_tuple)
    nplet_eval[dataset] = df

agg_cols   = ["F_score", "PR_AUC", "PR_AUC_inv"]
group_cols = ["order", "task", "measure", "nplet"]
df_agg     = {ds: df.groupby(group_cols, as_index=False)[agg_cols].mean()
              for ds, df in nplet_eval.items()}
for ds, df in df_agg.items():
    df["PR_AUC_signed"] = np.where(df["task"] == "c_gt_nr", df["PR_AUC"], df["PR_AUC_inv"])

top_nplets_dict = {}
df_top_list     = []
for dataset, df in df_agg.items():
    df_O = df[(df["measure"] == HOI_MEASURE) & df["order"].between(3, 9)].copy()
    df_O = df_O.sort_values(["order", "task", "PR_AUC_signed"], ascending=[True, True, False])
    top_O = df_O.groupby(["order", "task"]).head(N_TOP).reset_index(drop=True)
    for (k, task), sub in top_O.groupby(["order", "task"]):
        top_nplets_dict[(dataset, k, task)] = list(sub["nplet"])
    selected  = top_O[["order", "task", "nplet"]].drop_duplicates()
    df_full   = df.merge(selected, on=["order", "task", "nplet"], how="inner")
    df_full["dataset"] = dataset
    df_top_list.append(df_full)

df_top_all = pd.concat(df_top_list, ignore_index=True)
print(f"N_TOP={N_TOP} | entries: {len(top_nplets_dict)}")

In [ ]:
rows_box = []
for dataset in ["MA", "DBS"]:
    X_tensor, n_c, n_nr = build_X_for_dataset(dataset, all_covs)
    X_tensor = X_tensor.to(device=device, dtype=torch.float64)
    for polarity in ["c_gt_nr", "nr_gt_c"]:
        for order in range(3, 10):
            nplets = top_nplets_dict.get((dataset, order, polarity), [])
            if not nplets: continue
            nplets_t = torch.tensor(nplets, dtype=torch.long, device=device)
            measures = nplets_measures(X=X_tensor, covmat_precomputed=True,
                                        nplets=nplets_t, device=device, verbose=logging.WARNING)
            if measures.shape[0] != X_tensor.shape[0]:
                measures_np = measures.detach().cpu().numpy().transpose(1, 0, 2)
            else:
                measures_np = measures.detach().cpu().numpy()
            O_vals = measures_np[:, :, 2]
            for s in range(X_tensor.shape[0]):
                macro = "C" if s < n_c else "NR"
                for j in range(O_vals.shape[1]):
                    rows_box.append({"dataset": dataset, "polarity": polarity,
                                     "order": order, "macrostate": macro, "value": float(O_vals[s, j])})
df_box = pd.DataFrame(rows_box)
print(df_box.shape)

## PR-AUC vs order (Top-50 robustness check)

In [ ]:
summary = (
    df_top_all[df_top_all["order"].between(3, 9)]
    .groupby(["dataset", "measure", "task", "order"])["PR_AUC_signed"]
    .agg(["mean", "sem"])
    .reset_index()
)
summary["ci95"] = 1.96 * summary["sem"]

vir           = cm.get_cmap("magma")
colors        = vir(np.linspace(0.1, 0.9, 4))
color_dataset = {"MA": colors[0], "DBS": colors[2]}
ls_measure    = {"O": "-", "FC_mean_z": ":"}
mk_dataset    = {"MA": "o", "DBS": "s"}
measure_label = {"O": r"$\Omega$ (HOI)", "FC_mean_z": "Pairwise FC (Fisher z)"}
polarity_title = {
    "c_gt_nr": r"$\Omega_C > \Omega_{NR}$ — Top-50 robustness",
    "nr_gt_c": r"$\Omega_{NR} > \Omega_C$ — Top-50 robustness",
}

fig, axes = plt.subplots(2, 1, figsize=(9, 8), sharex=True, sharey=True)
legend_entries = {}

for ax, polarity in zip(axes, ["c_gt_nr", "nr_gt_c"]):
    for dataset in ["MA", "DBS"]:
        for measure in ["O", "FC_mean_z"]:
            sub = summary[
                (summary["dataset"] == dataset) &
                (summary["measure"] == measure) &
                (summary["task"]    == polarity)
            ]
            if sub.empty: continue
            h = ax.errorbar(sub["order"], sub["mean"], yerr=sub["ci95"],
                            color=color_dataset[dataset], linestyle=ls_measure[measure],
                            marker=mk_dataset[dataset], markersize=MSIZE, linewidth=LWIDTH,
                            capsize=3, alpha=0.9)
            legend_entries[(dataset, measure)] = h[0]
    ax.set_title(polarity_title[polarity], loc="left", fontsize=FONTSIZE + 1)
    ax.set_ylabel("PR AUC")
    ax.grid(axis="y", alpha=0.3, linestyle="--")
    sns.despine(ax=ax)

axes[1].set_xlabel("Order $k$")
handles = [legend_entries[(ds, m)] for m in ["O", "FC_mean_z"] for ds in ["MA", "DBS"]
           if (ds, m) in legend_entries]
lbls    = [f"{ds}, {measure_label[m]}" for m in ["O", "FC_mean_z"] for ds in ["MA", "DBS"]
           if (ds, m) in legend_entries]
axes[0].legend(handles, lbls, frameon=False)

plt.tight_layout()
plt.savefig(f"{results_path}/figS3_PRAUC_lines_N{N_TOP}.pdf", bbox_inches="tight")
plt.show()